# Component 2 — synthetic Colab runtime verification

Status: **prepared, not executed in Colab**. This notebook checks existing code; it does not train models, open dental images, load/download pretrained weights, approve labels or activate manifests.

Use a fresh standard CPU runtime (Hardware accelerator: None). Run cells in order. Upload `vision_colab_runtime_bundle_v1.zip` when cell 2 asks. No GitHub clone is used because these exact files are currently uncommitted. No Google Drive mount or package installation.

Model instances use `pretrained=False` (Torchvision `weights=None`). Two existing tests request pretrained metadata only to verify that uncached weights are blocked before any model is constructed or download attempted.

The six prior backend cases and two additional synthetic architecture cases must all pass with zero skips/errors before backend verification is claimed. A five-output random model is not a trained/validated five-class dental classifier.

In [ ]:
# Cell 1: use the existing Colab runtime; install nothing.
import importlib.util, json, platform, sys, traceback
from pathlib import Path
required = ["torch", "torchvision", "PIL", "numpy", "pytest"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
try:
    if missing:
        raise RuntimeError("Missing runtime packages: " + ", ".join(missing) + ". Stop; no auto-install.")
    import torch, torchvision, numpy as np, PIL, pytest
    from PIL import Image
    from torchvision.ops import nms
    probe = np.arange(4, dtype=np.float32)
    np.testing.assert_array_equal(torch.from_numpy(probe).numpy(), probe)
    assert nms(torch.tensor([[0., 0., 1., 1.]]), torch.tensor([1.]), 0.5).tolist() == [0]
    assert np.asarray(Image.new("RGB", (6, 4))).shape == (4, 6, 3)
    runtime_info = {"python": platform.python_version(), "torch": str(torch.__version__),
                    "torchvision": str(torchvision.__version__), "numpy": np.__version__,
                    "Pillow": PIL.__version__, "pytest": pytest.__version__, "test_device": "cpu",
                    "compatibility_probes": "imports, NumPy bridge, CPU torchvision op, RGB Pillow passed"}
    print(json.dumps(runtime_info, indent=2))
except Exception:
    message = traceback.format_exc()
    Path("/content/vision_compatibility_failure.txt").write_text(message)
    print(message)
    raise


In [ ]:
# Cell 2: upload only the supplied code-only ZIP; no Drive mount or Git clone.
from google.colab import files
import hashlib, io, os, tempfile, zipfile
uploaded = files.upload()
filename = "vision_colab_runtime_bundle_v1.zip"
assert set(uploaded) == {filename}, "Upload exactly the supplied code ZIP"
payload = uploaded[filename]
EXPECTED_SHA256 = "f590ff163ed2e8e20b3f07dc372052f7fad2b22ba2c5812f4fdd38cf94d78e9e"
assert hashlib.sha256(payload).hexdigest() == EXPECTED_SHA256, "Bundle mismatch; stop"
allowed_code = {"src/vision/model_pipeline.py", "tests/test_vision_model_pipeline.py",
                "tests/test_vision_colab_runtime.py"}
ROOT = Path(tempfile.mkdtemp(prefix="rp_component2_runtime_", dir="/content"))
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    assert len(archive.namelist()) == 4
    assert set(archive.namelist()) == allowed_code | {"runtime_source_manifest.json"}
    source_manifest = json.loads(archive.read("runtime_source_manifest.json"))
    assert source_manifest["source_branch"] == "vinuthan"
    assert set(source_manifest["files"]) == allowed_code
    for name in sorted(allowed_code):
        content = archive.read(name)
        assert hashlib.sha256(content).hexdigest() == source_manifest["files"][name]["sha256"]
        destination = ROOT / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        with destination.open("xb") as handle:
            handle.write(content)
(ROOT / "runtime_source_manifest.json").write_text(json.dumps(source_manifest, indent=2))
(ROOT / "runtime_versions.json").write_text(json.dumps(runtime_info, indent=2))
print("Verified code snapshot from vinuthan, including uncommitted work:", ROOT)
print(json.dumps(source_manifest, indent=2))


In [ ]:
# Cell 3: existing six backend tests, then two architecture-level Grad-CAM checks.
import subprocess, xml.etree.ElementTree as ET
selection = "actual_backend or uncached_pretrained or actual_autograd"
jobs = [("existing_backend", 6, ["tests/test_vision_model_pipeline.py", "-k", selection]),
        ("architecture_gradcam", 2, ["tests/test_vision_colab_runtime.py"])]
environment = dict(os.environ, PYTHONPATH=str(ROOT), PYTEST_DISABLE_PLUGIN_AUTOLOAD="1",
                   PYTHONDONTWRITEBYTECODE="1", OMP_NUM_THREADS="2", MKL_NUM_THREADS="2",
                   CUDA_VISIBLE_DEVICES="")
summaries = {}
for label, expected, targets in jobs:
    xml_path, log_path = ROOT / (label + ".xml"), ROOT / (label + ".log")
    command = [sys.executable, "-B", "-m", "pytest", "-q", "-rs", "--color=no",
               "-p", "no:cacheprovider", "--junitxml", str(xml_path), *targets]
    print("RUN", command)
    try:
        process = subprocess.run(command, cwd=ROOT, env=environment,
                                 capture_output=True, text=True, timeout=600)
        output = process.stdout + process.stderr
        log_path.write_text(output)
        print(output)
        if not xml_path.exists():
            raise RuntimeError("pytest produced no XML; retain log and investigate")
        cases = ET.parse(xml_path).getroot().findall(".//testcase")
        failed = sum(case.find("failure") is not None for case in cases)
        errors = sum(case.find("error") is not None for case in cases)
        skipped = sum(case.find("skipped") is not None for case in cases)
        passed = len(cases) - failed - errors - skipped
        summaries[label] = {"expected_tests": expected, "passed": passed, "failed": failed,
                            "errors": errors, "skipped": skipped, "returncode": process.returncode,
                            "verified": process.returncode == 0 and passed == expected
                                        and failed == errors == skipped == 0}
    except Exception:
        summaries[label] = {"verified": False, "execution_error": traceback.format_exc()}
        log_path.write_text(log_path.read_text() + summaries[label]["execution_error"] if log_path.exists()
                            else summaries[label]["execution_error"])
        print(summaries[label]["execution_error"])
    (ROOT / "verification_summary.json").write_text(json.dumps(summaries, indent=2))
    if not summaries[label]["verified"]:
        break  # Stop after a real failure/skip; do not weaken tests or auto-reinstall.
print(json.dumps(summaries, indent=2))
if len(summaries) == 2 and all(value["verified"] for value in summaries.values()):
    print("VERIFIED: 6 existing backend + 2 architecture Grad-CAM checks passed; no training/weights download.")
else:
    print("NOT VERIFIED: retain exact logs/XML. Run the evidence-download cell next.")


In [ ]:
# Cell 4: save actual evidence for review, including failures; no dataset content.
if "ROOT" in globals():
    evidence = ROOT / "vision_runtime_evidence.zip"
    names = ["runtime_versions.json", "runtime_source_manifest.json", "verification_summary.json",
             "existing_backend.log", "existing_backend.xml", "architecture_gradcam.log", "architecture_gradcam.xml"]
    with zipfile.ZipFile(evidence, "x", compression=zipfile.ZIP_DEFLATED) as archive:
        for name in names:
            path = ROOT / name
            if path.exists():
                archive.write(path, arcname=name)
    files.download(str(evidence))
else:
    from google.colab import files
    failure = Path("/content/vision_compatibility_failure.txt")
    assert failure.exists(), "No runtime execution evidence exists yet"
    files.download(str(failure))
